# Динамика Маркета и партнёра

Дневные данные за сентябрь 2026 года. Уникальные пользователи за разные дни пересекаются, поэтому их суммы нельзя трактовать как месячные уникальные аудитории. Короткий ряд не доказывает сезонность, причинность или устойчивый тренд.

## Как читать маршруты: уточнение для анализа

Нажатие на кнопку в карточке — это отдельное событие взаимодействия с карточкой. По предоставленным агрегатам нельзя определить, куда именно ведёт конкретная кнопка: к чекауту VK, в чат, на внешний сайт, на страницу партнёра или к другому действию. Поэтому нажатие не равно переходу к чекауту, оформлению заказа или оплате.

- **Маркет VK:** открытия карточки и нажатия описывают интерес к товару и действие пользователя, но следующий шаг не наблюдаем в разрезе назначений кнопки.
- **Партнёр 1:** нажатия в карточке партнёра и прямые переходы без открытия карточки — разные события и потенциально разные маршруты. Их нельзя складывать в одну воронку или считать одной когортой.
- **Оформленные заказы:** отдельная итоговая метрика канала. Без связки пользовательской сессии нельзя утверждать, какой именно кнопкой или переходом был вызван заказ.

Далее дневные отношения используются только как **интенсивность событий** и диагностический сигнал, а не как когортная конверсия.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path.cwd().parent / 'Data'
mv = 'Маркет Вконтакте'
p1 = 'Партнёр 1'
pd.set_option('display.float_format', '{:,.2f}'.format)

daily = pd.read_csv(data_path / 'Market_Partner_dynamics.csv', parse_dates=['Дата']).set_index('Дата').sort_index()
market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
total = pd.read_csv(data_path / 'Market_Total.csv').set_index('Период')
pd.Series({'Начало': daily.index.min(), 'Конец': daily.index.max(), 'Строк': len(daily), 'Дубли дат': daily.index.duplicated().sum(), 'Пропущенные даты': len(pd.date_range(daily.index.min(), daily.index.max()).difference(daily.index)), 'Пропуски в значениях': daily.isna().sum().sum()})

Начало                  2026-09-01 00:00:00
Конец                   2026-09-30 00:00:00
Строк                                    30
Дубли дат                                 0
Пропущенные даты                          0
Пропуски в значениях                      0
dtype: object

In [2]:
# Интерактивные визуализации
import plotly.graph_objects as go

def show_plotly(fig, title, x_title, y_title, height=500):
    fig.update_layout(
        title=dict(text=title), template='plotly_white', height=height,
        xaxis_title=x_title, yaxis_title=y_title,
        legend_title_text='Канал', margin=dict(l=70, r=30, t=80, b=90),
        font=dict(size=13), hoverlabel=dict(namelength=-1),
    )
    fig.show(config={'displaylogo': False, 'responsive': True})


## Проверка итогов

Суммируем только события, а не ежедневных уникальных пользователей. Расхождения показываются явно.

In [3]:
check = pd.DataFrame(index=['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке', 'Оформленные заказы', 'GMV, ₽'])
for channel in [mv, p1]:
    columns = [f'Показы товаров: {channel}', f'Открытия карточки: {channel}', f'Нажатия на кнопку: {channel}', f'Заказы: {channel}', f'GMV, ₽: {channel}']
    check[f'Сумма дней: {channel}'] = daily[columns].sum().to_numpy()
    check[f'Итог: {channel}'] = market.loc[check.index, channel].to_numpy()
    check[f'Разница: {channel}'] = check[f'Сумма дней: {channel}'] - check[f'Итог: {channel}']
check

,Сумма дней: Маркет Вконтакте,Итог: Маркет Вконтакте,Разница: Маркет Вконтакте,Сумма дней: Партнёр 1,Итог: Партнёр 1,Разница: Партнёр 1
Показы товаров,4777750023,"4,777,750,023.00",0.00,5729628790,"5,729,628,790.00",0.00
Открытия карточки товара,193512461,"193,512,461.00",0.00,86882822,"86,882,822.00",0.00
Нажатия на кнопку в карточке,5969063,"5,969,063.00",0.00,36626075,"36,626,075.00",0.00
Оформленные заказы,413740,"413,740.00",0.00,4927525,"4,927,525.00",0.00
"GMV, ₽",1564546740,"1,564,546,740.00",0.00,5119514684,"5,119,514,685.00",-1.00


In [4]:
pd.Series({'Средний DAU по дням': daily['DAU Маркета Вконтакте'].mean(), 'Средний DAU в итого': total.loc['DAU Маркета Вконтакте (среднее за день)', 'Значение'], 'Оплаты Маркета по дням': daily[f'Оплаченные заказы: {mv}'].sum(), 'Оплаты Маркета в итого': market.loc['Оплаченные заказы', mv]})

Средний DAU по дням      127,406.33
Средний DAU в итого      127,406.00
Оплаты Маркета по дням     4,903.00
Оплаты Маркета в итого     4,903.00
dtype: float64

## Распределения и выбросы

Коэффициент вариации = стандартное отклонение / среднее. Выбросы — наблюдения с |0,6745 × (значение − медиана) / MAD| > 3,5; если MAD равен нулю, метод не применяется. Это сигналы для проверки, а не ошибки, и они не исключаются.

In [5]:
stats = daily.describe().T
stats['Коэффициент вариации, %'] = stats['std'] / stats['mean'] * 100
stats[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'Коэффициент вариации, %']]

,mean,std,min,25%,50%,75%,max,"Коэффициент вариации, %"
DAU Маркета Вконтакте,"127,406.33","3,956.40","119,482.00","124,489.75","128,405.50","130,707.00","132,413.00",3.11
Видели товары (уник.),"23,901,299.27","777,948.96","22,203,574.00","23,433,692.75","23,948,617.50","24,593,862.75","24,972,021.00",3.25
Открывали карточку (уник.),"2,743,741.00","157,600.38","2,320,353.00","2,658,501.00","2,760,172.50","2,865,072.50","2,983,774.00",5.74
Поисковые запросы,"952,841.43","29,500.92","917,268.00","935,816.00","952,175.00","957,562.00","1,076,659.00",3.10
Поисковые запросы с открытием карточки,"175,194.20","8,290.55","160,155.00","170,614.50","173,421.50","178,739.00","204,671.00",4.73
Показы товаров: Маркет Вконтакте,"159,258,334.10","5,997,551.74","148,306,361.00","154,882,137.50","159,053,334.50","163,018,706.25","171,860,437.00",3.77
Показы товаров: Партнёр 1,"190,987,626.33","13,807,706.21","150,004,733.00","187,642,818.25","190,928,824.00","199,783,474.25","213,417,627.00",7.23
Открытия карточки: Маркет Вконтакте,"6,450,415.37","186,145.16","5,985,707.00","6,370,317.50","6,466,670.00","6,591,942.00","6,745,747.00",2.89
Открытия карточки: Партнёр 1,"2,896,094.07","280,432.76","2,190,784.00","2,698,387.50","2,894,552.50","3,068,946.00","3,436,298.00",9.68
Нажатия на кнопку: Маркет Вконтакте,"198,968.77","10,840.87","179,953.00","192,007.75","196,704.50","203,180.50","234,998.00",5.45


In [6]:
median = daily.median()
mad = (daily - median).abs().median().replace(0, np.nan)
robust_z = 0.6745 * (daily - median) / mad
outliers = robust_z.stack().rename('robust z').reset_index()
outliers = outliers[outliers['robust z'].abs() > 3.5].sort_values('robust z', key=abs, ascending=False)
outliers

,Дата,level_1,robust z
3,2026-09-01,Поисковые запросы,7.80
14,2026-09-01,"GMV, ₽: Маркет Вконтакте",5.27
4,2026-09-01,Поисковые запросы с открытием карточки,5.02
194,2026-09-11,"GMV, ₽: Маркет Вконтакте",4.46
9,2026-09-01,Нажатия на кнопку: Маркет Вконтакте,4.44
445,2026-09-25,Оплаченные заказы: Маркет Вконтакте,4.22
16,2026-09-01,Покупатели: Маркет Вконтакте,4.19
11,2026-09-01,Заказы: Маркет Вконтакте,4.03
6,2026-09-01,Показы товаров: Партнёр 1,-3.66


In [7]:
q1 = daily.quantile(0.25)
q3 = daily.quantile(0.75)
iqr = q3 - q1
iqr_flags = ((daily < q1 - 1.5 * iqr) | (daily > q3 + 1.5 * iqr)).sum().sort_values(ascending=False)
iqr_flags[iqr_flags > 0].rename('Количество IQR-выбросов')

Нажатия на кнопку: Партнёр 1              3
GMV, ₽: Маркет Вконтакте                  3
Открытия карточки: Маркет Вконтакте       2
Показы товаров: Партнёр 1                 2
Покупатели: Маркет Вконтакте              1
Заказы: Маркет Вконтакте                  1
Нажатия на кнопку: Маркет Вконтакте       1
Поисковые запросы с открытием карточки    1
Открывали карточку (уник.)                1
Поисковые запросы                         1
Оплаченные заказы: Маркет Вконтакте       1
Name: Количество IQR-выбросов, dtype: int64

1 сентября выделяется рядом показателей: поисковые запросы и заказы Маркета высокие, а показы партнёра низкие. Это дата начала учебного года, но эффект события без дополнительных данных не подтверждён. Дополнительно проверим 11 сентября по GMV и 25 сентября по оплатам; возможны и другие выбросы по таблице.

## Изменения за период и недельный срез

Сравниваются последние семь дней с 2–8 сентября: первая дата отдельно выделена как потенциальный выброс. Изменение относительно среднего первой недели; наклон линейного приближения нормирован на среднее всего ряда. Это описательные, а не причинные оценки.

In [8]:
trend = pd.DataFrame({'2–8 сентября': daily.loc['2026-09-02':'2026-09-08'].mean(), '24–30 сентября': daily.loc['2026-09-24':'2026-09-30'].mean()})
trend['Изменение, %'] = (trend['24–30 сентября'] / trend['2–8 сентября'] - 1) * 100
trend['Наклон, % от среднего в день'] = [np.polyfit(np.arange(len(daily)), daily[col], 1)[0] / daily[col].mean() * 100 for col in daily.columns]
trend

,2–8 сентября,24–30 сентября,"Изменение, %","Наклон, % от среднего в день"
DAU Маркета Вконтакте,"126,234.43","128,220.00",1.57,0.09
Видели товары (уник.),"23,224,590.57","24,416,620.00",5.13,0.26
Открывали карточку (уник.),"2,688,792.29","2,898,996.86",7.82,0.49
Поисковые запросы,"948,411.00","930,610.14",-1.88,-0.17
Поисковые запросы с открытием карточки,"170,711.57","172,095.00",0.81,-0.11
Показы товаров: Маркет Вконтакте,"161,773,349.14","155,844,827.14",-3.66,-0.25
Показы товаров: Партнёр 1,"189,930,034.71","195,316,601.71",2.84,0.38
Открытия карточки: Маркет Вконтакте,"6,363,813.71","6,605,005.71",3.79,0.23
Открытия карточки: Партнёр 1,"2,943,816.00","3,111,396.00",5.69,0.52
Нажатия на кнопку: Маркет Вконтакте,"198,699.29","189,791.71",-4.48,-0.34


### Интерактивно: ежедневные оформленные заказы
Все 30 наблюдений сохранены, включая 1 сентября. Масштабы каналов различаются; это не данные о переносе заказов между каналами.


In [9]:
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_scatter(x=daily.index, y=daily[f'Заказы: {channel}'], name=channel, mode='lines+markers',
                    line=dict(color=color, width=2), marker=dict(size=5),
                    hovertemplate='%{x|%d.%m.%Y}<br>%{fullData.name}: %{y:,.0f} заказов<extra></extra>')
fig.update_yaxes(rangemode='tozero')
fig.update_layout(hovermode='x unified')
show_plotly(fig, 'Оформленные заказы по дням', 'Дата', 'Оформленные заказы, шт.', 470)

### Интерактивно: ежедневный GMV
GMV показан в рублях без поправки на инфляцию и без вывода о причинах изменений.


In [10]:
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_scatter(x=daily.index, y=daily[f'GMV, ₽: {channel}'], name=channel, mode='lines+markers',
                    line=dict(color=color, width=2), marker=dict(size=5),
                    hovertemplate='%{x|%d.%m.%Y}<br>%{fullData.name}: %{y:,.0f} ₽<extra></extra>')
fig.update_yaxes(rangemode='tozero')
fig.update_layout(hovermode='x unified')
show_plotly(fig, 'GMV по дням', 'Дата', 'GMV, ₽', 470)

In [11]:
weekday = daily.copy()
weekday['День недели'] = weekday.index.day_of_week
weekday = weekday.groupby('День недели').median()
weekday.index = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
weekday[[f'Заказы: {mv}', f'Заказы: {p1}', f'GMV, ₽: {mv}', f'GMV, ₽: {p1}', 'Поисковые запросы']]

,Заказы: Маркет Вконтакте,Заказы: Партнёр 1,"GMV, ₽: Маркет Вконтакте","GMV, ₽: Партнёр 1",Поисковые запросы
Пн,"12,257.50","156,117.50","50,692,900.00","160,716,699.00","952,175.00"
Вт,"12,449.00","152,874.00","50,038,599.00","161,830,518.00","943,197.00"
Ср,"12,687.00","157,446.00","49,599,901.00","165,570,575.00","929,914.00"
Чт,"13,375.00","163,441.50","50,815,668.50","161,420,490.50","943,639.50"
Пт,"15,706.00","162,632.00","58,193,467.50","164,819,777.50","977,108.00"
Сб,"15,133.50","171,058.00","52,610,230.50","178,296,132.50","951,091.00"
Вс,"13,615.00","163,116.00","50,532,784.00","171,793,203.50","957,482.50"


За последние семь дней против 2–8 сентября GMV партнёра выше примерно на 47%, его заказы — на 17%; заказы Маркета ниже примерно на 3%. Сравнение выбранных недель чувствительно к выбросам, росту среднего чека и различию состава заказов. В каждом дне недели лишь 4–5 наблюдений.

## Дневные отношения показателей

Отношение событий за один день, например «заказы / открытия карточки» или «нажатия / открытия», не является конверсией одной когорты: в числителе и знаменателе могут быть разные пользователи, а заказ может быть получен после другого маршрута или в другой день.

Корректная трактовка:

- **нажатия / открытия** — интенсивность взаимодействия с карточкой;
- **заказы / открытия** — отношение итоговых заказов к объёму открытий за день, а не вероятность заказа после открытия;
- **заказы / нажатия** — также не конверсия, пока не подтверждены назначение кнопки и последовательность событий;
- **дневная динамика** — способ увидеть пики, просадки и совместное изменение показателей, но не причинность.

Для настоящей конверсии нужны когортные данные: идентификатор сессии или пользователя, timestamp событий, назначение кнопки, переход к оформлению и статус оплаты.

In [12]:
kpi = pd.DataFrame(index=daily.index)
for channel in [mv, p1]:
    kpi[f'Открытия / показы, %: {channel}'] = daily[f'Открытия карточки: {channel}'] / daily[f'Показы товаров: {channel}'] * 100
    kpi[f'Заказы / открытия, %: {channel}'] = daily[f'Заказы: {channel}'] / daily[f'Открытия карточки: {channel}'] * 100
    kpi[f'Средний чек, ₽: {channel}'] = daily[f'GMV, ₽: {channel}'] / daily[f'Заказы: {channel}']
kpi['Оплаченные / оформленные заказы Маркета, %'] = daily[f'Оплаченные заказы: {mv}'] / daily[f'Заказы: {mv}'] * 100
kpi['Доля Маркета в GMV двух каналов, %'] = daily[f'GMV, ₽: {mv}'] / (daily[f'GMV, ₽: {mv}'] + daily[f'GMV, ₽: {p1}']) * 100
kpi

,"Открытия / показы, %: Маркет Вконтакте","Заказы / открытия, %: Маркет Вконтакте","Средний чек, ₽: Маркет Вконтакте","Открытия / показы, %: Партнёр 1","Заказы / открытия, %: Партнёр 1","Средний чек, ₽: Партнёр 1","Оплаченные / оформленные заказы Маркета, %","Доля Маркета в GMV двух каналов, %"
Дата,,,,,,,,
2026-09-01,3.67,0.34,"3,267.33",1.46,5.89,877.30,0.78,37.04
2026-09-02,3.73,0.21,"3,909.51",1.50,6.52,832.30,1.65,27.46
2026-09-03,3.79,0.22,"4,241.23",1.55,6.28,824.08,1.27,29.20
2026-09-04,3.80,0.24,"3,443.24",1.51,5.83,842.36,1.20,27.15
2026-09-05,4.11,0.24,"3,518.12",1.63,5.34,922.55,0.91,24.84
2026-09-06,4.14,0.21,"3,734.23",1.61,5.00,940.78,1.08,24.24
2026-09-07,4.02,0.19,"3,988.50",1.51,4.98,917.31,1.27,26.60
2026-09-08,3.96,0.19,"4,019.49",1.52,4.91,992.30,1.41,26.17
2026-09-09,3.91,0.19,"3,930.10",1.51,5.34,952.68,1.51,25.14


### Интерактивно: средний чек по дням
Дневной средний чек — GMV за день, делённый на число оформленных заказов того же дня; изменение может отражать состав заказов.


In [13]:
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_scatter(x=kpi.index, y=kpi[f'Средний чек, ₽: {channel}'], name=channel, mode='lines+markers',
                    line=dict(color=color, width=2), marker=dict(size=5),
                    hovertemplate='%{x|%d.%m.%Y}<br>%{fullData.name}: %{y:,.2f} ₽<extra></extra>')
fig.update_layout(hovermode='x unified')
show_plotly(fig, 'Средний чек по дням', 'Дата', 'GMV / оформленные заказы, ₽', 470)

In [14]:
kpi.agg(['min', 'median', 'max']).T

,min,median,max
"Открытия / показы, %: Маркет Вконтакте",3.64,4.11,4.41
"Заказы / открытия, %: Маркет Вконтакте",0.18,0.21,0.34
"Средний чек, ₽: Маркет Вконтакте","3,267.33","3,893.03","4,454.32"
"Открытия / показы, %: Партнёр 1",1.40,1.51,1.63
"Заказы / открытия, %: Партнёр 1",4.83,5.76,6.56
"Средний чек, ₽: Партнёр 1",824.08,"1,072.43","1,163.90"
"Оплаченные / оформленные заказы Маркета, %",0.78,1.18,1.71
"Доля Маркета в GMV двух каналов, %",16.98,23.84,37.04


## Корреляции Спирмена

Коэффициенты описывают связь ежедневных рангов за 30 наблюдений; возможны тренд, общие факторы и выбросы. Корреляция не означает причинность, а слабая корреляция заказов каналов не доказывает отсутствие каннибализации.

In [2]:
selected = ['Поисковые запросы', 'Поисковые запросы с открытием карточки', 'DAU Маркета Вконтакте', f'Открытия карточки: {mv}', f'Нажатия на кнопку: {mv}', f'Заказы: {mv}', f'GMV, ₽: {mv}', f'Открытия карточки: {p1}', f'Нажатия на кнопку: {p1}', f'Заказы: {p1}', f'GMV, ₽: {p1}']
daily[selected].corr(method='spearman')

,Поисковые запросы,Поисковые запросы с открытием карточки,DAU Маркета Вконтакте,Открытия карточки: Маркет Вконтакте,Нажатия на кнопку: Маркет Вконтакте,Заказы: Маркет Вконтакте,"GMV, ₽: Маркет Вконтакте",Открытия карточки: Партнёр 1,Нажатия на кнопку: Партнёр 1,Заказы: Партнёр 1,"GMV, ₽: Партнёр 1"
Поисковые запросы,1.00,0.84,-0.24,-0.36,0.89,0.76,0.62,-0.39,-0.28,-0.43,-0.29
Поисковые запросы с открытием карточки,0.84,1.00,-0.32,-0.30,0.83,0.84,0.63,-0.23,-0.12,-0.22,0.02
DAU Маркета Вконтакте,-0.24,-0.32,1.00,0.55,-0.28,-0.60,-0.33,-0.01,-0.09,-0.25,-0.09
Открытия карточки: Маркет Вконтакте,-0.36,-0.30,0.55,1.00,-0.46,-0.51,-0.45,0.59,0.51,0.44,0.53
Нажатия на кнопку: Маркет Вконтакте,0.89,0.83,-0.28,-0.46,1.00,0.82,0.72,-0.51,-0.36,-0.39,-0.37
Заказы: Маркет Вконтакте,0.76,0.84,-0.60,-0.51,0.82,1.00,0.82,-0.21,-0.09,-0.05,-0.09
"GMV, ₽: Маркет Вконтакте",0.62,0.63,-0.33,-0.45,0.72,0.82,1.00,-0.32,-0.27,-0.10,-0.20
Открытия карточки: Партнёр 1,-0.39,-0.23,-0.01,0.59,-0.51,-0.21,-0.32,1.00,0.94,0.62,0.53
Нажатия на кнопку: Партнёр 1,-0.28,-0.12,-0.09,0.51,-0.36,-0.09,-0.27,0.94,1.00,0.57,0.49
Заказы: Партнёр 1,-0.43,-0.22,-0.25,0.44,-0.39,-0.05,-0.10,0.62,0.57,1.00,0.71


Связь заказов Маркета с поисковыми запросами и запросами с открытием карточки положительная; это аргумент проверить товарный поиск в эксперименте. Связь заказов двух каналов близка к нулю, но по ней нельзя оценить перенос заказов между каналами.